In [7]:
import os, sys
from os import listdir
from os.path import isfile, join
import numpy as np
from scipy import stats
from scipy.stats import f_oneway
from numpy import array
import pandas as pd
import geopandas as gp
# from gisutils import project
from shapely.geometry import Point
import statistics
import math
from math import log10, floor
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
import glob
from pyproj import Transformer

testingpguL={'col1':[-448.914,-440,-441.96,-448.914,-449.9,-450],'condition':[2,2,3,4,5]}
df=pd.DataFrame(data=testingpguL)

df.loc[:,'nearest10']=-9999
df['roundgz'] =-9999
df['roundgzint'] =-9999
df['numstring'] =-9999
df['ld'] =-9999
df['ldan'] =-9999

df['roundgz'] = df['col1'].round()
df['roundgzint'] = df['roundgz'].astype(int)
df['nearest10'] = df['col1'].round(-1)
df['numstring'] = df['roundgzint'].astype(str)
df['ld'] = df['numstring'].str[-1]
df['ldan'] = df['ld'].astype(int)

# Calculate rgzbotm based on conditions
    df['rgzbotm'] = np.where(df['col1'] > 0,
                            df['roundgzint'] - df['ldan'],
                            df['roundgzint'] + df['ldan'])
    
    condition1 = (df['dep'] >= -450) & (df['ld'].isin([8, 9])) & (df['dep'] > 0)

    # Assign values based on conditions
    df.loc[condition1, 'topelv1'] = df['nearest10int']
    df.loc[condition1, 'botelv1'] = df['topelv1'] - 5


print('roundgz:',df['roundgz'][0])
print('roundgzint:',df['roundgzint'][0])
print('nearest10:',df['nearest10'][0])
print('numstring:',df['numstring'][0])
print('ld:',df['ld'][0])
print('ldan:',df['ldan'][0])

In [8]:
# Get list of pathline files in the specified directory
#pathline_dir = r'C:\Users\chaugh\Documents\micheal_g_files'
#pathline_dir = r'C:\github\map_gwage\MERASwd\Scripts\meras\testFILZ'
#pathline_dir = r'C:\github\map_gwage\MERASwd\Scripts\meras'
#pathline_dir = r'C:\github\map_gwage\MERASwd\Models\meras_2.2\layers_cal_wt_1.00\meras1\remaining'

pathline_dir = r'C:\github\map_gwage\MERASwd\Models\meras_2.2\layers_cal_wt_1.00\meras1\Shellmound\output'
#pathline_dir = r'C:\Users\mgratzer\Documents\dissertation\chapter3\pathlineFiles\trunc'
#meras_files = glob.glob(os.path.join(pathline_dir, '*.mppth'))
meras_files = glob.glob(os.path.join(pathline_dir, '*.csv'))
#pathline_dir = '../../../pathlineFiles8294'
#meras_files = [f for f in os.listdir(pathline_dir) if os.path.isfile(os.path.join(pathline_dir, f))]

# Initialize variables
sitenos = [f[-24:-9] for f in meras_files]  # Extract site number from file names
print('sitenos:',sitenos)
num_files = len(meras_files)
print('Number of files:', num_files)

# Create DataFrame from file names and site numbers
merasdf = pd.DataFrame({'indexpl': range(num_files), 'namepl': meras_files, 'siteno': sitenos})

# Loop through each pathline file
for idx, row in merasdf.iterrows():
    filename = row['namepl']
    #filename = 'meras_2.2_volume_zones[323757090515301].mppth'
    #filename = 'meras_2.2_volume_zones[320233091395501].mppth'
    siteno = row['siteno']
    print('Processing file:', filename)

    # Define column names and read pathline file
    #column_names = ['Particle ID', 'Particle group', 'Time Point Index', 'Cumulative Time Step', 'Tracking Time','Global X', 'Global Y', 'Global Z', 'Layer', 'Row', 'Column', 'Grid','Local X', 'Local Y', 'Local Z', 'Line Segment Index']
    file_path = os.path.join(pathline_dir, filename)
    #df = pd.read_csv(file_path, skiprows=3, header=None, delim_whitespace=True, names=column_names)
    #df = pd.read_csv(file_path, skiprows=3, header=None, sep='\s+', names=column_names)
    df = pd.read_csv(file_path)
    dfshp=df.shape
    print(dfshp)
    
    # Convert coordinates from local to EPSG:5070
    #xoff, yoff = 178389, 938511.6
    #df['x_5070'] = xoff + df['Global X'] * 12 * 2.54 * 0.01
    #df['y_5070'] = yoff + df['Global Y'] * 12 * 2.54 * 0.01

    # Reproject to EPSG:4326
    #coords_5070 = np.array(list(zip(df['x_5070'], df['y_5070'])))
    #coords_4326 = project(coords_5070, 'epsg:5070', 'epsg:4326')
    #df['x_4326'], df['y_4326'] = coords_4326[:, 0], coords_4326[:, 1]
    #Transform the coordinates from 5070 to 4326
    transformer = Transformer.from_crs("EPSG:5070", "EPSG:4326", always_xy=True)
    df['x_4326'], df['y_4326'] = transformer.transform(df['x_5070'].values, df['y_5070'].values)

    print('Reprojection completed')

    # Initialize new columns with default values
    # Initialize new columns with default values
    #default_values = {'roundgz': -9999,'roundgzint':-9999,'nearest10': -9999,'nearest10int':-9999,'numstring': 'xx', 'ld': 'xx', 'ldan': -9999,
    #    'rgzbotm': -9999, 'topelv1': -9999, 'botelv1': -9999, 'topelv2': -9999, 'botelv2': -9999,
    #    'elevationSlice1': 'xx', 'elevationSlice2': 'xx', 'res_ohm_m1': -9999, 'meanresdenom': -9999,'hmeanresWell':-9999,'dfms':-9999,
    #    'log10res':-9999,'hmeanlog10resWell':-9999,'dfms_logres':-9999,'distance traveled': -9999, 'CumulativeDistanceTraveledIndex': -9999,
    #    'CumulativeDistanceTraveled': -9999, 'distance traveled within cell': -9999,
    #    'CumulativeDistanceTraveledwinCellIndex': -9999, 'CumulativeDistanceTraveledwinCell': -9999,
    #    'mpk':0,'mplogk':0,'mpsnk':0,'mpsnlogk':0,'logNormalityWell_logresiss':'xx','cellIndex':-9999,'logNormalityWell':'xx','numresiss':-9999,'grp_dfms':-9999,'resistivity':-9999}
    #for col, value in default_values.items():
    #    df[col] = value
    #default_values = {
    #         'siteno':'xx','pids':'xx','dfmr':-9999,'dfmsr':-9999,'roundgz': -9999,'roundgzint':-9999,
    #         'nearest10': -9999,'nearest10int':-9999,'numstring': 'xx', 'ld': 'xx', 'ldan': -9999,
    #         'rgzbotm': -9999, 'topelv1': -9999, 'botelv1': -9999, 'topelv2': -9999, 'botelv2': -9999,
    #         'elevationSlice1': 'xx', 'elevationSlice2': 'xx', 'res_ohm_m1': -9999, 'res_ohm_m2': -9999,
    #         'log10res': -9999,'thickness':-9999,'mpk': 0}
    default_values = {'siteno':'xx','pids':'xx','dfmr':-9999,'dfmsr':-9999,'roundgz': -9999,'roundgzint':-9999,
                      'nearest10': -9999,'nearest10int':-9999,'numstring': 'xx', 'ld': 'xx', 'ldan': -9999,
                      'rgzbotm': -9999, 'topelv1': -9999, 'botelv1': -9999, 'topelv2': -9999, 'botelv2': -9999,
                      'elevationSlice1': 'xx', 'elevationSlice2': 'xx', 'res_ohm_m1': -9999,'res_ohm_m2': -9999,
                      'log10res': -9999,'thickness':-9999,'mpk': 0 'meanresdenom': -9999,'meanresdenom2':-9999,
                      'hmeanresWell':-9999,'dfms':-9999,'log10res':-9999,'hmeanlog10resWell':-9999,
                      'dfms_logres':-9999,'distance traveled': -9999,'CumulativeDistanceTraveledIndex': -9999,
                      'CumulativeDistanceTraveled': -9999, 'distance traveled within cell': -9999,
                      'CumulativeDistanceTraveledwinCellIndex': -9999, 'CumulativeDistanceTraveledwinCell': -9999,
                      'mpk':0,'mplogk':0,'mpsnk':0,'mpsnlogk':0,'logNormalityWell_logresiss':'xx','cellIndex':-9999,
                      'logNormalityWell':'xx','numresiss':0,'grp_dfms':-9999,'resistivity':-9999,'RasterValue2':-9999}
    #default_values = {'roundgz': -9999,'roundgzint':-9999,'nearest10': -9999,'nearest10int':-9999,
    #                       'numstring': 'xx', 'ld': 'xx', 'ldan': -9999,
    #         'rgzbotm': -9999, 'topelv1': -9999, 'botelv1': -9999, 'topelv2': -9999, 'botelv2': -9999,
    #         'elevationSlice1': 'xx', 'elevationSlice2': 'xx', 'res_ohm_m1': -9999, 'meanresdenom': -9999,
    #                       'meanresdenom2':-9999,'hmeanresWell':-9999,'dfms':-9999,
    #         'log10res':-9999,'hmeanlog10resWell':-9999,'dfms_logres':-9999,'distance traveled': -9999,
    #                       'CumulativeDistanceTraveledIndex': -9999,
    #         'CumulativeDistanceTraveled': -9999, 'distance traveled within cell': -9999,
    #         'CumulativeDistanceTraveledwinCellIndex': -9999, 'CumulativeDistanceTraveledwinCell': -9999,
    #         'mpk':0,'mplogk':0,'mpsnk':0,'mpsnlogk':0,'logNormalityWell_logresiss':'xx','cellIndex':-9999,
    #                       'logNormalityWell':'xx','numresiss':1,'grp_dfms':-9999,'resistivity':-9999}
    for col, value in default_values.items():
        df[col] = value

    df['siteno'] = siteno

    # Start elevation extraction logic
    print('Start extraction.')

    # Convert Global Z to meters and compute rounding values
    #df['dep'] = df['Global Z'] * 12 * 2.54 * 0.01
    df['roundgz'] = df['dep'].round()
    df['roundgzint'] = df['roundgz'].astype(int)
    df['nearest10'] = df['dep'].round(-1)
    df['nearest10int'] = df['nearest10'].astype(int)
    df['numstring'] = df['roundgzint'].astype(str)
    df['ld'] = df['numstring'].str[-1]
    df['ldan'] = df['ld'].astype(int)
    df['grp_dfms'] = df['grp_dfms'].astype(float)
    df['res_ohm_m1'] = df['res_ohm_m1'].astype(float)

    # Calculate rgzbotm based on conditions
    #df['rgzbotm'] = np.where(df['dep'] > 0,df['roundgzint'] - df['ldan'],df['roundgzint'] + df['ldan'])#rgzbotm is the multiple of 10 <= roundgzint.
    
    #     a=['dep','roundgz','roundgzint','nearest10','nearest10int','numstring','ld','rgzbotm']
    #     for apple in a:
    #         print('{}:'.format(apple),df[apple][0])
    #     sys.exit()
    #condition1 = (df['dep'] >= -450) &(df['dep'] <= 300) & (df['ld'].isin(['8','9'])) & (df['dep'] > 0)
    #condition1 = (df['dep'] >= 0) &(df['dep'] <= 130) & (df['ld'].isin(['8','9']))
    condition1=(df['dep']>=0)&(df['dep']<=130)&(df['ld'].isin(['0','2','4','6','8']))&(df['dep']>df['roundgz'])

    # Assign values based on conditions
    df.loc[condition1, 'topelv1'] = df['roundgzint']+2
    df.loc[condition1, 'botelv1'] = df['roundgzint']

    #condition2 = (df['dep'] >= -450)&(df['dep'] <= 300) & (df['ld'].isin(['8','9'])) & (df['dep'] < 0)
    condition2=(df['dep']>=0)&(df['dep']<=130)&(df['ld'].isin(['0','2','4','6','8']))&(df['dep']<=df['roundgz'])

    df.loc[condition2, 'botelv1'] = df['roundgzint']-2
    df.loc[condition2, 'topelv1'] = df['roundgzint']

    #condition3 = (df['dep'] >= -450)&(df['dep'] <= 300) & (df['ld'].isin(['1','2'])) & (df['dep'] > 0)
    condition3=(df['dep']>=0)&(df['dep']<=130)&(df['ld'].isin(['1','3','5','7','9']))

    df.loc[condition3, 'botelv1'] = df['roundgzint']-1
    df.loc[condition3, 'topelv1'] = df['roundgzint']+1

    #     condition4 = (df['dep'] >= -450)&(df['dep'] <= 300) & (df['ld'].isin(['1','2'])) & (df['dep'] < 0)

    #     df.loc[condition4, 'topelv1'] = df['nearest10int']
    #     df.loc[condition4, 'botelv1'] = df['topelv1'] - 5

    #     condition5 = (df['dep'] >= -450)&(df['dep'] <= 300) & (df['ld'] == '0') & (df['dep'] < df['roundgzint']) & (df['dep'] > 0)

    #     df.loc[condition5, 'topelv1'] = df['nearest10int']
    #     df.loc[condition5, 'botelv1'] = df['topelv1'] - 5

    #     condition6 = (df['dep'] >= -450)&(df['dep'] <= 300) & (df['ld'] == '0') & (df['dep'] < df['roundgzint']) & (df['dep'] < 0)

    #     df.loc[condition6, 'topelv1'] = df['nearest10int']
    #     df.loc[condition6, 'botelv1'] = df['topelv1'] - 5

    #     condition7 = (df['dep'] >= -450)&(df['dep'] <= 300) & (df['ld'] == '0') & (df['dep'] > df['roundgzint']) & (df['dep'] > 0)

    #     df.loc[condition7, 'botelv1'] = df['nearest10int']
    #     df.loc[condition7, 'topelv1'] = df['botelv1'] + 5

    #     condition8 = (df['dep'] >= -450)&(df['dep'] <= 300) & (df['ld'] == '0') & (df['dep'] > df['roundgzint']) & (df['dep'] < 0)

    #     df.loc[condition8, 'botelv1'] = df['nearest10int']
    #     df.loc[condition8, 'topelv1'] = df['botelv1'] + 5

    #     condition9=(df['dep']>=-450)&(df['dep'] <= 300)&(df['ld']=='0')&(df['dep']==df['roundgzint'])&(df['dep']>0)

    #     df.loc[condition9, 'topelv1'] = df['nearest10int']
    #     df.loc[condition9, 'botelv1'] = df['topelv1'] - 5
    #     df.loc[condition9, 'botelv2'] = df['nearest10int']
    #     df.loc[condition9, 'topelv2'] = df['botelv2'] + 5

    #     condition10 = (df['dep'] >= -450)&(df['dep'] <= 300) & (df['ld'] == '0') & (df['dep'] == df['roundgzint']) & (df['dep'] < 0)

    #     df.loc[condition10, 'botelv1'] = df['nearest10int']
    #     df.loc[condition10, 'topelv1'] = df['botelv1'] + 5
    #     df.loc[condition10, 'topelv2'] = df['nearest10int']
    #     df.loc[condition10, 'botelv2'] = df['topelv2'] - 5

    #     condition11=(df['dep']>=-450)&(df['dep'] <= 300)&(df['ld'].isin(['6','7']))&(df['dep']>0)

    #     df.loc[condition11, 'topelv1'] = df['nearest10int']
    #     df.loc[condition11, 'botelv1'] = df['topelv1'] - 5

    #     condition12=(df['dep']>=-450)&(df['dep']<=300)&(df['ld'].isin(['6','7']))&(df['dep']<0)

    #     df.loc[condition12,'botelv1']=df['nearest10int']
    #     df.loc[condition12,'topelv1']=df['botelv1']+5

    #     condition13=(df['dep']>=-450)&(df['dep']<=300)&(df['ld'].isin(['3','4']))&(df['dep']>0)

    #     df.loc[condition13, 'botelv1'] = df['nearest10int']
    #     df.loc[condition13, 'topelv1'] = df['botelv1'] + 5

    #     condition14 = (df['dep'] >= -450)&(df['dep'] <= 300)&(df['ld'].isin(['3','4']))&(df['dep'] < 0)

    #     df.loc[condition14, 'topelv1'] = df['nearest10int']
    #     df.loc[condition14, 'botelv1'] = df['topelv1'] - 5

    #     condition15=(df['dep']>=-450)&(df['dep']<=300)&(df['ldan']==5)&(df['dep']==df['roundgzint'])&(df['dep'] > 0)

    #     df.loc[condition15, 'botelv1'] = df['rgzbotm']
    #     df.loc[condition15, 'topelv1'] = df['botelv1'] + 5
    #     df.loc[condition15, 'botelv2'] = df['roundgz']
    #     df.loc[condition15, 'topelv1'] = df['botelv2'] + 5

    #     condition16 = (df['dep'] >= -450)&(df['dep'] <= 300) & (df['ldan'] == 5) & (df['dep'] == df['roundgzint']) & (df['dep'] < 0)

    #     df.loc[condition16, 'botelv1'] = df['roundgzint']
    #     df.loc[condition16, 'topelv1'] = df['botelv1'] + 5
    #     df.loc[condition16, 'topelv2'] = df['roundgzint']
    #     df.loc[condition16, 'botelv2'] = df['topelv2'] - 5

    #     condition17 = (df['dep'] >= -450)&(df['dep'] <= 300) & (df['ldan'] == 5) & (df['dep'] < df['roundgzint']) & (df['dep'] > 0)

    #     df.loc[condition17, 'topelv1'] = df['roundgzint']
    #     df.loc[condition17, 'botelv1'] = df['topelv1'] - 5

    #     condition18 = (df['dep'] >= -450)&(df['dep'] <= 300)& (df['ldan'] == 5) & (df['dep'] < df['roundgzint']) & (df['dep'] < 0)

    #     df.loc[condition18, 'topelv1'] = df['roundgzint']
    #     df.loc[condition18, 'botelv1'] = df['topelv1'] - 5

    #     condition19 = (df['dep'] >= -450)&(df['dep'] <= 300)& (df['ldan'] == 5) & (df['dep'] > df['roundgzint']) & (df['dep'] > 0)

    #     df.loc[condition19, 'botelv1'] = df['roundgzint']
    #     df.loc[condition19, 'topelv1'] = df['botelv1'] + 5

    #     condition20 = (df['dep'] >= -450)&(df['dep'] <= 300)& (df['ldan'] == 5) & (df['dep'] > df['roundgzint']) & (df['dep'] < 0)

    #     df.loc[condition20, 'botelv1'] = df['roundgzint']
    #     df.loc[condition20, 'topelv1'] = df['botelv1'] + 5

    #df.loc[(df['botelv1']!=-9999)&(df['topelv1']!=-9999),'elevationSlice1'] = 'elv_res_' + df['topelv1'].astype(str) + '_' + df['botelv1'].astype(str) + 'm.tif'
    df.loc[(df['botelv1']!=-9999)&(df['topelv1']!=-9999),'elevationSlice1'] = 'SHMDdep_res_' + df['botelv1'].astype(str) + '_' + df['topelv1'].astype(str) + 'm.tif'
    #df['elevationSlice1'] = 'elv_res_' + df['topelv1'].astype(str) + '_' + df['botelv1'].astype(str) + 'm.tif'

    #df['elevationSlice2'] = 'elv_res_' + df['topelv2'].astype(str) + '_' + df['botelv2'].astype(str) + 'm.tif'
    #df['tiffName1']=df['elevationSlice1']
    numress=0
    numlogress=0
    meanresdenom=0
    meanlogresdenom=0
    for index, row in df.iterrows():
        if row['elevationSlice1']!='xx':
            tiffName1 = row['elevationSlice1']
            #tiffName2 = row['elevationSlice2']

            #with rasterio.open(os.path.join(r"C:\Users\chaugh\Documents\micheal_g_files\MY_TIFFS\MY_TIFFS", tiffName1)) as src1:
            #Looks like src2 is not used in the code...
            # if (tiffName2!=- 'xx'):
            #     src2 = rasterio.open(os.path.join...)
            with rasterio.open(os.path.join(r"C:\github\AEM\examples\OneDrive_1_2-24-2022\SHMD_TIFFS", tiffName1)) as src1:
                #raster_value1 = next(src1.sample([(row['x_4326'], row['y_4326'])]))[0]
                raster_value2 = next(src1.sample([(row['x_5070'], row['y_5070'])]))[0]
                #with rasterio.open(os.path.join(r"C:\github\AEM\examples\OneDrive_1_2-24-2022\MY_TIFFS", tiffName1)) as src1:
                #raster_value = next(src1.sample([(row['x_4326'], row['y_4326'])]))[0]
                #print("Raster Value: ", raster_value)
                #df.at[index, 'RasterValue1'] = raster_value1
                df.at[index, 'RasterValue2'] = raster_value2
                #print(df['RasterValue1'])
                #print(df['RasterValue2'])
                #sys.exit()

    df.loc[df['RasterValue2']>(10**30),'RasterValue2']=-9999
    df['res_ohm_m1'] = df['res_ohm_m1'].astype(float)
    for index, row in df.iterrows():
        if row['elevationSlice1']!='xx':
            df.at[index, 'res_ohm_m1'] = df['RasterValue2'][index]
            #df.loc[df['res_ohm_m1']!=-9999,'res_ohm_m1'] = 10**(df['res_ohm_m1'])
    df.loc[df['res_ohm_m1'>=0,'resistivity']=10**(df['res_ohm_m1'])
    #df.loc[df['res_ohm_m1']!=-9999,'numresiss']=1
    df.loc[df['resistivity']>0,'numresiss']=1
    df['meanresdenom'] = df['meanresdenom'].astype(float)
    #df.loc[df['res_ohm_m1']!=-9999,'meanresdenom']=1/(df['res_ohm_m1'])
    #df['meanresdenom']=1/(df['resistivity'])
    #df.loc[df['RasterValue2']!=0,'meanresdenom']=1/(df['resistivity'])
    df.loc[df['resistivity']>0,'meanresdenom']=1/(df['resistivity'])
    #df['rgzbotm'] = np.where(df['dep'] > 0,df['roundgzint'] - df['ldan'],df['roundgzint'] + df['ldan'])#rgzbotm is the multiple of 10 <= roundgzint.
    
    #df['meanresdenom2']=np.where(df['resistivity']>0,1/(df['resistivity']),np.nan)
    #df['meanresdenom']=1/(df['resistivity'])
    df.loc[df['res_ohm_m1']>=0,'numlogresiss']=1
    #df.loc[:,'numlogresiss']=1
    df['log10res'] = df['log10res'].astype(float)
    #df.loc[df['res_ohm_m1']>0,'log10res']=np.log10(df['res_ohm_m1'])
    df.loc[df['res_ohm_m1']>=0,'log10res']=df['res_ohm_m1']
    #df['log10res']=np.log10(df['resistivity'])
    df.loc[:,'meanlogresdenom']=-9999
    df['meanlogresdenom'] = df['meanlogresdenom'].astype(float)
    #df.loc[df['res_ohm_m1']>0,'meanlogresdenom']=1/(df['log10res'])
    #df['meanlogresdenom']=1/(df['log10res'])
    df.loc[df['log10res']>=0,'meanresdenom']=1/(df['log10res'])
    #             for x in ['res_ohm_m1','numresiss','meanresdenom']:
    #                 df.loc[df[x]==-9999,x]=np.nan
    #             if row['res_ohm_m1']!=-9999:
    #                 numress=numress+1
    #                 meanresdenom=meanresdenom+(1/(row['res_ohm_m1']))
    #             if row['res_ohm_m1']>0:
    #                 numlogress=numlogress+1
    #                 row['log10res'] = np.log10(row['res_ohm_m1'])
    #                 meanlogresdenom=meanlogresdenom+(1/(row['log10res']))
    #     df.loc[df['res_ohm_m1']==-9999,'res_ohm_m1']=np.nan
    #df['log10res']=log10(df['res_ohm_m1'])
    #     df['log10res'] = np.where(df['res_ohm_m1'] > 0, np.log10(df['res_ohm_m1']), np.nan)
    #     df.loc[:,'numresiss']=numress
    #     df.loc[:,'numlogresiss']=numlogress
    #df.to_csv("temp_file.csv")
    df=df.replace(-9999,np.nan)
    #for x in ['res_ohm_m1','numresiss','meanresdenom','numlogresiss','meanlogresdenom']:
    #    df.loc[df[x]==-9999,x]=np.nan
    print("Extraction complete for file: ", filename)
    print("Thank You!")
    #sys.exit()

    #Only change I made from here on was to the handling of sitenos and the construction of the well_summ_parts dataframe.
    #meanres=np.nanmean(df['res_ohm_m1'])  # arithmetic average of all resistivities for all particles tracked from the well.
    numress=np.nansum(df['numresiss'])
    print('numress:',numress)
    #meanresdenom=np.nansum(df['meanresdenom2'])
    meanresdenom=np.nansum(df['meanresdenom'])
    print('meanresdenom:',meanresdenom)
    meanres=numress/meanresdenom  # harmonic average of all resistivities for all particles tracked from the well.
    df['hmeanresWell'] = df['hmeanresWell'].astype(float)
    df['dfms'] = df['dfms'].astype(float)
    df.loc[:,'hmeanresWell']=meanres
    #df=df.replace(np.nan,-9999)
    #df['dfms']=np.where(df['res_ohm_m1']!=-9999,((df['res_ohm_m1']-meanres)**2),np.nan)
    df['dfms']=(df['resistivity']-meanres)**2
    #for x in ['hmeanresWell','dfms']:
    #    df.loc[df[x]==-9999,x]=np.nan
    VARresWell = (np.nansum(df['dfms']))/(numress - 1)
    df.loc[:,'VARresWell']=VARresWell
    print('VARresWell:',VARresWell)
    #df.to_csv('compare2pguL.csv')
    #exit
    #sys.exit()
    #df['dfms']=(df['res_ohm_m1']-meanres)**2
    # dfms_df = grpdf[['dfms']]
    # # Save the dfms DataFrame to a CSV file
    # dfms_df.to_csv(f'dfms_column_{name}.csv', index=False)
    
    numlogress=np.nansum(df['numlogresiss'])
    print('numlogress:',numlogress)
    meanlogresdenom=np.nansum(df['meanlogresdenom'])
    print('meanlogresdenom:',meanlogresdenom)
    meanlog10res=numlogress/meanlogresdenom  # harmonic average of all resistivities for all particles tracked from the well.
    #meanlog10res=np.nanmean(df['log10res'])
    df['hmeanlog10resWell'] = df['hmeanlog10resWell'].astype(float)
    df.loc[:,'hmeanlog10resWell']=meanlog10res
    df['dfms_logres'] = df['dfms_logres'].astype(float)
    #df['dfms_logres'] = np.where(df['log10res']!=-9999,((df['log10res']-meanlog10res)**2), np.nan)
    df['dfms_logres'] = (df['log10res']-meanlog10res)**2
    for x in ['hmeanlog10resWell','dfms_logres']:
        df.loc[df[x]==-9999,x]=np.nan
    VARlogresWell = (np.nansum(df['dfms_logres']))/(numlogress - 1)
    #print('VARlogresWell:',VARlogresWell)
    #df.to_csv('compafe.csv')
    #exit
    #sys.exit()
    
    medianresWell=statistics.median(df['resistivity'])
    minresWell=min(df['resistivity'])
    maxresWell=max(df['resistivity'])
    medianlogresWell=statistics.median(df['log10res'])
    minlogresWell=min(df['log10res'])
    maxlogresWell=max(df['log10res'])

    # Goodness of fit test to see if log resistivities are normally distributed in which case the overall distribution is lognormal. Here, this is being applied to all particles tracked from the well.
    loc,scale = np.nanmean(df['log10res']), np.std(df['log10res'], ddof=1)
    cdf = stats.norm(loc, scale).cdf
    res=stats.ks_1samp(df['log10res'], cdf)
    ###print(res)
    if res.pvalue<.05:
        print('not log normal')
        lnwlr='not log normal'
        df.loc[:,'logNormalityWell_logresiss']=lnwlr
    else:
        print('log normal')
        lnwlr='log normal'
        df.loc[:,'logNormalityWell_logresiss']=lnwlr
	
    #######################
    # Now group df by PID and calculate	variance and effective resistivity for each particle. Then calculate average of variances and variance of effective resistivities for each well. [Each name is a PID (particle). Each group is the df for that particle. Numrecs is the number of locations recorded along the particle's path. Each q is a location recorded along the particle's path.]

    PIDgrp=df.groupby('Particle ID')

    sites=[]
    VARs=[]#before looping to the next particle, store this particle's variance of resistivities along its flowpath in the list VARs.
    number_of_locs=[]
    numnans=[]
    pathlengths=[]
    RESeffs=[]#before looping to the next particle, store this particle's RESeff in the list RESeffs.
    snRESeffs=[]
    logRESeffs=[]
    snlogRESeffs=[]
    logVARs=[]#before looping to the next particle, store this particle's variance of log resistivities along its flowpath in the list logVARs.
    VARs_sn=[]
    VARs_snlogres=[]
    numsrespart=[]
    VEXT=[]
    VRs=[]
    cvrs=[]
    DFMERs=[]
    DFMSERs=[]
    DFMVRs=[]
    DFMSVRs=[]
    DFMcVRs=[]
    DFMScVRs=[]
    logNormality=[]
           
    for name,group in PIDgrp:#for each particle:
        print('Name:',name)
        grpshp=group.shape#This might not be numlocs because We know that some locations were recorded twice if they were on cell faces, edges, or corners. We must check for consecutive identical locations. On second thought, it will likely take too much time to correct this so I think for now We accpt this as a limitation.
        numlocs=grpshp[0]  # numlocs = the number of locations recorded along the particle's flowpath.
        #print('numlocs:',numlocs)
        numlocsWell=0
        INDX=range(numlocs)
        
        ##SORT FIRST##
        group = group.sort_values(by=['Cumulative Time Step','Tracking Time'],ascending=[False,True])
        #group.to_csv(f'particle_{name}_v2.csv', index=False)
        grpdf = group.reset_index(drop=True)
        #print('columns:',grpdf.columns.tolist())
        #sys.exit()
        grpdf=grpdf.replace(-9999,np.nan)
        maxz=max(grpdf['dep'])
        minz=min(grpdf['dep'])
        #print('minz:',minz)
        EV=maxz-minz
        #VEXT.append(EV)
        grpdf=grpdf.drop_duplicates(subset=['Particle ID','Global X','Global Y','Global Z'])
        grpshp=grpdf.shape
        numlocspart=grpshp[0]
        #grpdf.loc[:,'nnres']=0
        #grpdf.loc[grpdf['res_ohm_m1']!=-9999,'nnres']=1
        grpdf=grpdf.replace(np.nan,-9999)
        grpdf=grpdf.loc[grpdf['resistivity']!=-9999]
        grpshp=grpdf.shape
        numrespart=grpshp[0]
        #print('numrespart:',numrespart)
        #numsrespart.append(numrespart)
        if numrespart>0:
            grpdf=grpdf.replace(-9999,np.nan)
            smeanresdenom=np.nansum(grpdf['meanresdenom2'])
            #print('smeanresdenom:',smeanresdenom)
            #meanlog10res=numlogress/meanlogresdenom  # harmonic average of all resistivities for all particles tracked from the well.
            #meanlog10res=np.nanmean(df['log10res'])
            #df.loc[:,'hmeanlog10resWell']=meanlog10res

            #print('numrespart:',numrespart)
            #sys.exit()
            #grpdf=pd.DataFrame(data=group, index=INDX)
            #grpdf.to_csv(f'grpdf_for_{name}.csv', index=False)
            #print('res_ohm_m1: ', grpdf['res_ohm_m1'])
            #smeanres=np.nanmean(grpdf['res_ohm_m1'])  # Average resistivity recorded along a particle's path
            smeanres=numrespart/smeanresdenom
            print("smeanres: ", smeanres)
            #smeanlogres=np.nanmean(grpdf['log10res'])  # Average log resistivity recorded along a particle's path
            #print('smeanlogres: ', smeanlogres)
            #One thing I've noticed: there is often only 1 value in res_ohm_m1, so the mean of that column is equal to the values, so in the line below, res_ohm_m1 - smeanres = 0, hence why
            #grpdf['grp_dfms'] = grpdf['grp_dfms'].astype(float)
            grpdf['grp_dfms']=(grpdf['resistivity']-smeanres)**2
            # dfms_df = grpdf[['dfms']]

            # # Save the dfms DataFrame to a CSV file
            # dfms_df.to_csv(f'dfms_column_{name}.csv', index=False)

            #grpdf['dfms_logres']=(grpdf['log10res']-smeanlogres)**2
            #print('grpdf[dfms_logres]: ', grpdf['dfms_logres'])
            #grpdf.to_csv(f'grpdf_v2_{name}.csv', index=False)
            # Calculate standard normal resistivity
            #VAR=(math.fsum(grpdf['dfms']))/(numlocs-1)  # Variance of raw resistivities along one particle's flowpath.
            VAR = (np.nansum(grpdf['grp_dfms']))/(numrespart - 1)
            #print("sum: ", np.nansum(grpdf['dfms']))
            #print("Numlocs: ", numlocs)
            #print('VAR: ', VAR)
            #sys.exit()
            stdres=VAR**.5
            grpdf['grpressn']=(grpdf['resistivity']-smeanres)/stdres
            ###########################++++++++++++++++++++++++++++++
            grpnumlogress=np.nansum(grpdf['numlogresiss'])
            print('numlogress:',grpnumlogress)
            grpmeanlogresdenom=np.nansum(grpdf['meanlogresdenom'])
            print('meanlogresdenom:',grpmeanlogresdenom)
            grpmeanlog10res=grpnumlogress/grpmeanlogresdenom  # harmonic average of all resistivities for all particles tracked from the well.
            #meanlog10res=np.nanmean(df['log10res'])
            grpdf.loc[:,'hmeanlog10resPart']=grpmeanlog10res
            grpdf=grpdf.replace(np.nan,-9999)
            grpdf['grpdfms_logres'] = np.where(grpdf['log10res']!=-9999,((grpdf['log10res']-grpmeanlog10res)**2), np.nan)
            for x in ['hmeanlog10resPart','grpdfms_logres']:
                grpdf.loc[grpdf[x]==-9999,x]=np.nan
            grpdf.loc[grpdf['resistivity']==0,'resistivity']=np.nan
            VARlogresPart = (np.nansum(grpdf['grpdfms_logres']))/(grpnumlogress - 1)
            #VARlogres=(math.fsum(grpdf['dfms_logres']))/(numlocs-1)  # Variance of log resistivities along one particle's flowpath.
            #VARlogres = (np.nansum(grpdf['dfms_logres'])) / (numlocs - 1)
            #print('VARlogres: ', VARlogres)
            #stdlogres=(VARlogres)**.5
            #grpdf['logressn']=(grpdf['log10res']-smeanlogres)/stdlogres

            sitenolist=grpdf['siteno'].tolist()
            #print('len(sitenolist):',len(sitenolist))
            lx=grpdf['Local X'].tolist()
            #print('len(lx):',len(lx))
            #         print('grpdf[Local X][0]:',grpdf['Local X'][0])
            #         print('grpdf[Local X][0]:',grpdf['Local X'][1])
            #         print('lx[0]:',lx[0])
            #         print('lx[1]:',lx[1])
            ly=grpdf['Local Y'].tolist()
            lz=grpdf['Local Z'].tolist()
            gx=grpdf['Global X'].tolist()
            gy=grpdf['Global Y'].tolist()
            gz=grpdf['Global Z'].tolist()
            dt=grpdf['distance traveled'].tolist()
            cdti=grpdf['CumulativeDistanceTraveledIndex'].tolist()
            cdt=grpdf['CumulativeDistanceTraveled'].tolist()
            dtwc=grpdf['distance traveled within cell'].tolist()
            cdtwci=grpdf['CumulativeDistanceTraveledwinCellIndex'].tolist()
            cdtwc=grpdf['CumulativeDistanceTraveledwinCell'].tolist()
            ci=grpdf['cellIndex'].tolist()
            layer=grpdf['Layer'].tolist()
            row=grpdf['Row'].tolist()
            column=grpdf['Column'].tolist()
            mpk=grpdf['mpk'].tolist()
            mplk=grpdf['mplogk'].tolist()
            mpsnk=grpdf['mpsnk'].tolist()
            #mpsnlk=grpdf['mpsnlogk'].tolist()
            rom1=grpdf['resistivity'].tolist()
            ltr=grpdf['log10res'].tolist()
            rsn=grpdf['grpressn'].tolist()
            #lrsn=grpdf['logressn'].tolist()

            dt[0]=0
            cdti[0]=0
            cdt[0]=0
            dtwc[0]=0
            cdtwci[0]=0
            cdtwc[0]=0
            ci[0]=0
            #print('numlocs:',numlocs)

            # This third part of the script is for calculating distances between sequential recorded particle locations.
            if numrespart>=2:
                r=np.linspace(1,numrespart-1,numrespart-1).astype(int)
                for q in r:
                    #print('q:',q)
                    if (lx[q]!=lx[q-1])|(ly[q]!=ly[q-1])|(lz[q]!=lz[q-1]):

                        dt[q]=(((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**.5
                        cdti[q]=cdti[q-1]+1
                        cdt[q]=dt[q]+cdt[q-1]
                        #########################################################
                        if (layer[q]==layer[q-1])&(row[q]==row[q-1])&(column[q]==column[q-1]):
                            dtwc[q]=(((gx[q]-gx[q-1])**2)+((gy[q]-gy[q-1])**2)+((gz[q]-gz[q-1])**2))**(.5)
                            cdtwci[q]=cdtwci[q-1]+1
                            cdtwc[q]=dtwc[q]+cdtwc[q-1]
                            ci[q]=ci[q-1]

                        else:
                            dtwc[q]=0
                            cdtwci[q]=0
                            cdtwc[q]=0
                            ci[q]=ci[q-1]+1
                        mpk[q]=(dtwc[q])/(rom1[q])
                        mplk[q]=(dtwc[q])/(ltr[q])
                        mpsnk[q]=(dtwc[q])/(rsn[q])
                        #mpsnlk[q]=(dtwc[q])/(lrsn[q])
                        #print('Distances for particle {} calculated!'.format(name))
                #print('Distances for all particles calculated!')
                cumDisPart=max(cdt)

                #RESeffdenom=math.fsum(mpk)
                RESeffdenom=np.nansum(mpk)

                if RESeffdenom!=0:
                    RESeff=(max(cdt))/RESeffdenom  # effective resistivity of one particle's flowpath.
                else:
                    #print('else condition met for RESeff')
                    RESeff=np.nan

                #logRESeffdenom=math.fsum(mplk)
                logRESeffdenom=np.nansum(mplk)
                if logRESeffdenom!=0:
                    logRESeff=(max(cdt))/logRESeffdenom  # effective resistivity of one particle's flowpath.
                else:
                    #print('else condition met for logRESeff')
                    logRESeff=np.nan

                ###########################++++++++++++++++++++++++++++++
                #calculate variance of standard normal resistivities along this particle's flowpath.
                grpdf=grpdf.join(pd.DataFrame({'grpressn2':rsn}))
                smeansnres=np.nanmean(rsn)
                grpdf['dfms_snres']=(grpdf['grpressn2']-smeansnres)**2
                #VARsnres=(math.fsum(grpdf['dfms_snres']))/(numlocs-1)
                VARsnres = (np.nansum(grpdf['dfms_snres']))/(numlocs - 1)
                stdsnres=(VARsnres)**.5
                #print('VARsnres: ', VARsnres)
                ###########################++++++++++++++++++++++++++++++
                #calculate variance of standard normal log resistivities along this particle's flowpath.
                #             grpdf=grpdf.join(pd.DataFrame({'lrsn2':lrsn}))
                #             smeansnlogres=np.nanmean(grpdf['lrsn2'])
                #             grpdf['dfms_logsnres']=(grpdf['lrsn2']-smeansnlogres)**2
                #             #VARsnlogres=(math.fsum(grpdf['dfms_logsnres']))/(numlocs-1)
                #             VARsnlogres=(np.nansum(grpdf['dfms_logsnres']))/(numlocs-1)
                #             stdsnlogres=(VARsnlogres)**.5
                #print('VARsnlogres: ', VARsnlogres)

                #calculate effective res of standard normal resistivities along this particle's flowpath.
                grpdf=grpdf.join(pd.DataFrame({'mpsnk2':mpsnk,'cdt2':cdt}))
                #snRESeffdenom=math.fsum(grpdf['mpsnk2']) 
                snRESeffdenom=np.nansum(grpdf['mpsnk2']) 
                if snRESeffdenom!=0:
                    snRESeff=(max(grpdf['cdt2']))/snRESeffdenom#effective resistivity of one particle's flowpath.
                else:
                    #print('else condition met for snRESeff')
                    snRESeff=np.nan

                #calculate effective res of standard normal log resistivities along this particle's flowpath.
                #grpdf=grpdf.join(pd.DataFrame({'mpsnlk2':mpsnlk}))
                #snlogRESeffdenom=math.fsum(grpdf['mpsnlk2'])
                #             snlogRESeffdenom=np.nansum(grpdf['mpsnlk2'])
                #             if snlogRESeffdenom!=0:
                #                 snlogRESeff=(max(grpdf['cdt2']))/snlogRESeffdenom#effective resistivity of one particle's flowpath.
                #             else:
                #                 #print('else condition met for snlogRESeff')
                #                 snlogRESeff=np.nan

                pathlengths.append(cumDisPart)
                RESeffs.append(RESeff)#before looping to the next particle, store this particle's RESeff in the list RESeffs.
                snRESeffs.append(snRESeff)
                logRESeffs.append(logRESeff)
                VEXT.append(EV)
                numsrespart.append(numrespart)
                #snlogRESeffs.append(snlogRESeff)
                VARs.append(VAR)#before looping to the next particle, store the variance of the resistivities along this particle's flowpath in the list VARs.
                logVARs.append(VARlogresPart)#before looping to the next particle, store this particle's variance of log resistivities along its flowpath in the list logVARs.
                VARs_sn.append(VARsnres)
                #VARs_snlogres.append(VARsnlogres)
                sites.append(name)#before looping to the next particle, store this particle's ID in the list sites.
                number_of_locs.append(numlocs)# numlocs = the number of locations recorded along the particle's flowpath
                numlocsWell=numlocsWell+numlocspart
                #grpdf.loc[:,numlocsWell]=numlocsWell
                #print('Calculations for particle {} completed!'.format(name))
                #print("VARs: ", VARs)
    print('Calculations for all particles completed!')
    Numero=len(numsrespart)
    Nombre=len(pathlengths)
    Numbr=len(VEXT)
    Numbrvr=len(VARs)
    #I added thes three lines to get around an error I was getting (length of the lists was not the same for creating the data frame 'well_summ_parts'), 
    #you'll want to double check that this is treating the 'siteno's the right way
    #these three lines take the 'siteno' list that is created from the grpdf and filters out all NA values, then repeats the non-NA siteno for the length of the other lists used to create the dataframe
    siteno_series = pd.Series(sitenolist)
    # Find the first non-NaN value
    first_non_na = siteno_series.dropna().iloc[0] if not siteno_series.dropna().empty else None
    # Repeat the first non-NaN value 100 times
    new_sitenos = [first_non_na] * len(sites) if first_non_na is not None else []
    
    #d={'sitenumber':new_sitenos,'Particle ID':sites,'numlocs':number_of_locs,'LENGTHtot':pathlengths,'vertext':VEXT,'var_res':VARs,'var_logres':logVARs,'var_res_sn':VARs_sn,'var_snlogres':VARs_snlogres,'RESeff':RESeffs,'snRESeff':snRESeffs,'logRESeff':logRESeffs,'snlogRESeff':snlogRESeffs,'numrespart':numsrespart}
    #well_summ_parts=pd.DataFrame(data=d)
    
    #     d={'sitenumber':new_sitenos,'Particle ID':sites,'numlocs':number_of_locs,'LENGTHtot':pathlengths,'vertext':VEXT,'var_res':VARs,'var_logres':logVARs,'var_res_sn':VARs_sn,'var_snlogres':VARs_snlogres,'RESeff':RESeffs,'snRESeff':snRESeffs,'logRESeff':logRESeffs,'numrespart':numsrespart}
    #     well_summ_parts=pd.DataFrame(data=d)
    lists=[new_sitenos,sites,number_of_locs,pathlengths,VEXT,VARs,logVARs,VARs_sn,RESeffs,snRESeffs,logRESeffs,numsrespart]
    for x in lists:
        print('len{}'.format(x),len(x))
    d={'sitenumber':new_sitenos,'Particle ID':sites,'numlocs':number_of_locs,'LENGTHtot':pathlengths,'vertext':VEXT,'var_res':VARs,'var_logres':logVARs,'var_res_sn':VARs_sn,'RESeff':RESeffs,'snRESeff':snRESeffs,'logRESeff':logRESeffs,'numrespart':numsrespart}
    well_summ_parts=pd.DataFrame(data=d)
    
    #d={'sitenumber':new_sitenos,'Particle ID':sites,'numlocs':number_of_locs,'var_res':VARs,'var_logres':logVARs,'var_res_sn':VARs_sn,'var_snlogres':VARs_snlogres,'RESeff':RESeffs,'snRESeff':snRESeffs,'logRESeff':logRESeffs,'snlogRESeff':snlogRESeffs}
    #well_summ_parts=pd.DataFrame(data=d)
    nwell_summ_parts=well_summ_parts.replace(np.nan,-9999)
    nrpwell_summ_parts=nwell_summ_parts.loc[nwell_summ_parts['numrespart']!=-9999]
    nrpshp=nrpwell_summ_parts.shape
    Numero=nrpshp[0]
    nrpwell_summ_parts=nrpwell_summ_parts.replace(-9999,np.nan)
    sumnrp=np.nansum(nrpwell_summ_parts['numrespart'])
    ameannrp=np.nanmean(nrpwell_summ_parts['numrespart'])
    nrpwell_summ_parts['dfms_nrp']=(nrpwell_summ_parts['numrespart']-ameannrp)**2
    #VARsnlogres=(np.nansum(grpdf['dfms_logsnres']))/(numlocs-1)
    varnumrespart=(np.nansum(nrpwell_summ_parts['dfms_nrp']))/(Numero-1)
    mediannrp=statistics.median(nrpwell_summ_parts['numrespart'])
    minnrp=min(nrpwell_summ_parts['numrespart'])
    maxnrp=max(nrpwell_summ_parts['numrespart'])
    well_summ_parts.loc[:,'sumnrp']=sumnrp
    well_summ_parts.loc[:,'ameannrp']=ameannrp
    well_summ_parts.loc[:,'varnumrespart']=varnumrespart
    well_summ_parts.loc[:,'mediannrp']=mediannrp
    well_summ_parts.loc[:,'minnrp']=minnrp
    well_summ_parts.loc[:,'maxnrp']=maxnrp
    
    well_summ_parts.loc[:,'lnwlr']=lnwlr
    
    #     nwell_summ_parts=well_summ_parts.replace(np.nan,-9999)
    #     nrpwell_summ_parts=nwell_summ_parts.loc[nwell_summ_parts['numrespart']!=-9999]
    #     nrpshp=nrpwell_summ_parts.shape
    #     Numero=nrpshp[0]
    #nrpwell_summ_parts=nrpwell_summ_parts.replace(-9999,np.nan)
    sumcdtp=np.nansum(nrpwell_summ_parts['LENGTHtot'])
    ameancdtp=np.nanmean(nrpwell_summ_parts['LENGTHtot'])
    #grpdf['dfms_cdtp']=(well_summ_parts['LENGTHtot']-ameancdtp)**2
    nrpwell_summ_parts['dfms_cdtp']=(nrpwell_summ_parts['LENGTHtot']-ameancdtp)**2
    varLENGTHtot=(np.nansum(nrpwell_summ_parts['dfms_cdtp']))/(Numero-1)
    mediancdtp=statistics.median(nrpwell_summ_parts['LENGTHtot'])
    mincdtp=min(nrpwell_summ_parts['LENGTHtot'])
    maxcdtp=max(nrpwell_summ_parts['LENGTHtot'])
    well_summ_parts.loc[:,'sumcdtp']=sumcdtp
    well_summ_parts.loc[:,'ameancdtp']=ameancdtp
    well_summ_parts.loc[:,'varLENGTHtot']=varLENGTHtot
    well_summ_parts.loc[:,'mediancdtp']=mediancdtp
    well_summ_parts.loc[:,'mincdtp']=mincdtp
    well_summ_parts.loc[:,'maxcdtp']=maxcdtp
    
    #     nwell_summ_parts=well_summ_parts.replace(np.nan,-9999)
    #     nrpwell_summ_parts=nwell_summ_parts.loc[nwell_summ_parts['numrespart']!=-9999]
    #     nrpshp=nrpwell_summ_parts.shape
    #     Numero=nrpshp[0]
    #     nrpwell_summ_parts=nrpwell_summ_parts.replace(-9999,np.nan)
    sumvep=np.nansum(nrpwell_summ_parts['vertext'])
    ameanvep=np.nanmean(nrpwell_summ_parts['vertext'])
    nrpwell_summ_parts['dfms_vep']=(nrpwell_summ_parts['vertext']-ameanvep)**2
    varvertext=(np.nansum(nrpwell_summ_parts['dfms_vep']))/(Numbr-1)
    medianvep=statistics.median(well_summ_parts['vertext'])
    minvep=min(nrpwell_summ_parts['vertext'])
    maxvep=max(nrpwell_summ_parts['vertext'])
    well_summ_parts.loc[:,'sumvep']=sumvep
    well_summ_parts.loc[:,'ameanvep']=ameanvep
    well_summ_parts.loc[:,'varvertext']=varvertext
    well_summ_parts.loc[:,'medianvep']=medianvep
    well_summ_parts.loc[:,'minvep']=minvep
    well_summ_parts.loc[:,'maxvep']=maxvep
    
    #     nwell_summ_parts=well_summ_parts.replace(np.nan,-9999)
    #     nrpwell_summ_parts=nwell_summ_parts.loc[nwell_summ_parts['numrespart']!=-9999]
    #     nrpshp=nrpwell_summ_parts.shape
    #     Numero=nrpshp[0]
    #     nrpwell_summ_parts=nrpwell_summ_parts.replace(-9999,np.nan)
    sumvrp=np.nansum(nrpwell_summ_parts['var_res'])
    ameanvrp=np.nanmean(nrpwell_summ_parts['var_res'])
    nrpwell_summ_parts['dfms_vrp']=(nrpwell_summ_parts['var_res']-ameanvrp)**2
    varvar_res=(np.nansum(nrpwell_summ_parts['dfms_vrp']))/(Numero-1)
    medianvrp=statistics.median(nrpwell_summ_parts['var_res'])
    minvrp=min(nrpwell_summ_parts['var_res'])
    maxvrp=max(nrpwell_summ_parts['var_res'])
    well_summ_parts.loc[:,'sumvrp']=sumvrp
    well_summ_parts.loc[:,'ameanvrp']=ameanvrp
    well_summ_parts.loc[:,'varvar_res']=varvar_res
    well_summ_parts.loc[:,'medianvrp']=medianvrp
    well_summ_parts.loc[:,'minvrp']=minvrp
    well_summ_parts.loc[:,'maxvrp']=maxvrp
    
    nrpwell_summ_parts=nrpwell_summ_parts.replace(np.nan,-9999)
    rewell_summ_parts=nrpwell_summ_parts.loc[nrpwell_summ_parts['RESeff']!=-9999]
    numparts=len(rewell_summ_parts['RESeff'])
    print('number of particles:',numparts)
    well_summ_parts.loc[:,'numlocsWell']=numlocsWell
    fracwres=numress/numlocsWell
    well_summ_parts.loc[:,'fracwres']=fracwres
    #df['rgzbotm'] = np.where(df['dep'] > 0,df['roundgzint'] - df['ldan'],df['roundgzint'] + df['ldan'])#
    #well_summ_parts['inversereseff']=1/(well_summ_parts['RESeff'])
    well_summ_parts=well_summ_parts.replace(np.nan,-9999)
    well_summ_parts['inversereseff']=np.where(well_summ_parts['RESeff']!=-9999,1/(well_summ_parts['RESeff']),-9999)
    well_summ_parts=well_summ_parts.replace(-9999,np.nan)
    hmeanreseffdenom=np.nansum(well_summ_parts['inversereseff'])
    hmeanreseff=numparts/hmeanreseffdenom  # harmonic average of all effective resistivities for all particles tracked from the well.
    ameanreseff=np.nanmean(rewell_summ_parts['RESeff'])
    well_summ_parts.loc[:,'hmeanreseffW_bad']=hmeanreseff
    well_summ_parts.loc[:,'ameanreseffWel']=ameanreseff
    medianreseffWell=statistics.median(rewell_summ_parts['RESeff'])
    minreseffWell=min(rewell_summ_parts['RESeff'])
    maxreseffWell=max(rewell_summ_parts['RESeff'])
    
    well_summ_parts=well_summ_parts.replace(np.nan,-9999)
    well_summ_parts['inverselogreseff']=np.where(well_summ_parts['RESeff']>0,1/(well_summ_parts['logRESeff']),-9999)
    #well_summ_parts['inverselogreseff']=1/(well_summ_parts['logRESeff'])
    well_summ_parts=well_summ_parts.replace(-9999,np.nan)
    hmeanlogreseffdenom=np.nansum(well_summ_parts['inverselogreseff'])
    nrpwell_summ_parts=nrpwell_summ_parts.replace(np.nan,-9999)
    lrewell_summ_parts=nrpwell_summ_parts.loc[nrpwell_summ_parts['logRESeff']!=-9999]
    numparts=len(lrewell_summ_parts['logRESeff'])
    hmeanlogreseff=numparts/hmeanlogreseffdenom  # harmonic average of all resistivities for all particles tracked from the well.
    well_summ_parts.loc[:,'hmeanlreW_bad']=hmeanlogreseff
    ameanlogreseff=np.nanmean(lrewell_summ_parts['logRESeff'])
    well_summ_parts.loc[:,'ameanlreW']=ameanlogreseff
    medianlogreseffWell=statistics.median(lrewell_summ_parts['logRESeff'])
    minlogreseffWell=min(lrewell_summ_parts['logRESeff'])
    maxlogreseffWell=max(lrewell_summ_parts['logRESeff'])
    
    well_summ_parts.loc[:,'medianresWell']=medianresWell
    well_summ_parts.loc[:,'minresWell']=minresWell
    well_summ_parts.loc[:,'maxresWell']=maxresWell
    well_summ_parts.loc[:,'medianlogresWell']=medianlogresWell
    well_summ_parts.loc[:,'minlogresWell']=minlogresWell
    well_summ_parts.loc[:,'maxlogresWell']=maxlogresWell
    well_summ_parts.loc[:,'medianreseffWell']=medianreseffWell
    well_summ_parts.loc[:,'minreseffWell']=minreseffWell
    well_summ_parts.loc[:,'maxreseffWell']=maxreseffWell
    well_summ_parts.loc[:,'medianlogreseffWell']=medianlogreseffWell
    well_summ_parts.loc[:,'minlogreseffWell']=minlogreseffWell
    well_summ_parts.loc[:,'maxlogreseffWell']=maxlogreseffWell
    
    well_summ_parts=well_summ_parts.replace(np.nan,-9999)
    nlwell_summ_parts=well_summ_parts.loc[well_summ_parts['numlocs']!=-9999]
    numparts=len(nlwell_summ_parts['numlocs'])
    
    meannumlocs=np.nanmean(well_summ_parts['numlocs'])
    well_summ_parts['dfms_numlocs']=np.where(well_summ_parts['numlocs']!=-9999,(well_summ_parts['numlocs']-meannumlocs)**2,-9999)
    #well_summ_parts['dfms_numlocs']=(well_summ_parts['numlocs']-meannumlocs)**2
    #VARsnlogres=(math.fsum(grpdf['dfms_logsnres']))/(numlocs-1)
    well_summ_parts=well_summ_parts.replace(-9999,np.nan)
    varnumlocs=(np.nansum(well_summ_parts['dfms_numlocs']))/(numparts-1)
    mediannumlocs=statistics.median(nlwell_summ_parts['numlocs'])
    minnumlocs=min(nlwell_summ_parts['numlocs'])
    maxnumlocs=max(nlwell_summ_parts['numlocs'])
    #after looping through all particles for a given well, test whether the effective resistivities are log normal.
    #meanlog10reseff=np.nanmean(logRESeffs)
    #well_summ_parts.loc[:,'meanlog10reseff']=meanlog10reseff
    well_summ_parts.loc[:,'meannumlocs']=meannumlocs
    well_summ_parts.loc[:,'varnumlocs']=varnumlocs
    well_summ_parts.loc[:,'mediannumlocs']=mediannumlocs
    well_summ_parts.loc[:,'minnumlocs']=minnumlocs
    well_summ_parts.loc[:,'maxnumlocs']=maxnumlocs
    
    well_summ_parts.loc[:,'logNormalityWell']='xx'

    # Goodness of fit test for effective resistivities of all the particles tracked from the well, to see if they have a lognormal distribution.
    #loc,scale = meanlog10reseff, np.std(logRESeffs, ddof=1)
    #loc,scale = ameanlreW, np.std(logRESeffs, ddof=1)
    loc,scale = ameanlogreseff, np.std(logRESeffs, ddof=1)
    cdf = stats.norm(loc, scale).cdf
    res=stats.ks_1samp(logRESeffs, cdf)
    ###print(res)
    if res.pvalue<.05:
        print('not log normal')
        well_summ_parts.loc[:,'logNormalityWell']='not log normal'
    else:
        print('log normal')
        well_summ_parts.loc[:,'logNormalityWell']='log normal'
    numparts=len(sites)
    print('number of particles:',numparts)
    numRESeffs=len(RESeffs)
    print('number of effective resistivities:',numRESeffs)
    print('\t')
    well_summ_partsshp=well_summ_parts.shape
    print('well_summ_parts:',well_summ_partsshp)

    # Calculating average variance of resistivity along a flowpath.
    smeanvar=np.nanmean(well_summ_parts['var_res'])
    ###########################++++++++++++++++++++++++++++++
    smeanlogvar=np.nanmean(well_summ_parts['var_logres'])
    smeanvar_sn=np.nanmean(well_summ_parts['var_res_sn'])
    #smeanvar_logsn=np.nanmean(well_summ_parts['var_snlogres'])
    
    well_summ_parts.loc[:,'numresiss']=numress#total number of resistivities associated with the well in question
    well_summ_parts.loc[:,'numlogresiss']=numlogress
    well_summ_parts.loc[:,'hmeanresWell']=meanres
    well_summ_parts.loc[:,'hmeanlog10resWell']=meanlog10res
    well_summ_parts.loc[:,'VARresWell']=VARresWell
    well_summ_parts.loc[:,'VARlogresWell']=VARlogresWell
    well_summ_parts.loc[:,'medianresWell']=medianresWell
    well_summ_parts.loc[:,'minresWell']=minresWell
    well_summ_parts.loc[:,'maxresWell']=maxresWell
    well_summ_parts.loc[:,'medianlogresWell']=medianlogresWell
    well_summ_parts.loc[:,'minlogresWell']=minlogresWell
    well_summ_parts.loc[:,'maxlogresWell']=maxlogresWell

    ameanreseff=np.nanmean(well_summ_parts['RESeff'])
    well_summ_parts.loc[:,'avg_resEff']=ameanreseff
    ameansnreseff=np.nanmean(well_summ_parts['snRESeff'])
    well_summ_parts.loc[:,'avg_snresEff']=ameansnreseff
    ameanlogreseff=np.nanmean(well_summ_parts['logRESeff'])
    well_summ_parts.loc[:,'avg_logresEff']=ameanlogreseff
    #ameansnlogreseff=np.nanmean(well_summ_parts['snlogRESeff'])
    #well_summ_parts.loc[:,'avg_snlogresEff']=ameansnlogreseff

    # Calculating variance of effective resistivities 
    well_summ_parts['dfms_reseff']=(well_summ_parts['RESeff']-ameanreseff)**2
    #VAR=(math.fsum(well_summ_parts['dfms_reseff']))/(numRESeffs-1)
    VAR=(np.nansum(well_summ_parts['dfms_reseff']))/(numRESeffs-1)
    well_summ_parts.loc[:,'VAR_resEff']=VAR#variance of effective resistivities of flowpaths to the well.

    # Calculating variance of sn effective res
    well_summ_parts['dfms_reseff_sn']=(well_summ_parts['snRESeff']-ameansnreseff)**2
    #VARsn=(math.fsum(well_summ_parts['dfms_reseff_sn']))/(numRESeffs-1)
    VARsn=(np.nansum(well_summ_parts['dfms_reseff_sn']))/(numRESeffs-1)
    well_summ_parts.loc[:,'VAR_snresEff']=VARsn#variance of sn effective resistivities of flowpaths to the well.

    #Calculating variance of effective log res
    well_summ_parts['dfms_logreseff']=(well_summ_parts['logRESeff']-ameanlogreseff)**2
    #VARlog10res=(math.fsum(well_summ_parts['dfms_logreseff']))/(numRESeffs-1)
    VARlog10res=(np.nansum(well_summ_parts['dfms_logreseff']))/(numRESeffs-1)
    well_summ_parts.loc[:,'VAR_logresEff']=VARlog10res#variance of effective log resistivities of flowpaths to the well.

    # Calculating variance of effective sn log res
    #     well_summ_parts['dfms_snlogreseff']=(well_summ_parts['snlogRESeff']-ameansnlogreseff)**2
    #     #VARsnlog10res=(math.fsum(well_summ_parts['dfms_snlogreseff']))/(numRESeffs-1)
    #     VARsnlog10res=(np.nansum(well_summ_parts['dfms_snlogreseff']))/(numRESeffs-1)
    #     well_summ_parts.loc[:,'VAR_snlogresEff']=VARsnlog10res#variance of sn effective log resistivities of flowpaths to the well.

    well_summ_parts.loc[:,'avgVar_res']=smeanvar#average of variances of resistivities along flowpaths to the well
    well_summ_parts.loc[:,'avgVar_logres']=smeanlogvar
    well_summ_parts.loc[:,'avgVar_snres']=smeanvar_sn
    #well_summ_parts.loc[:,'avgVar_snlogres']=smeanvar_logsn
    well_summ_parts.loc[:,'avgRES_well']=meanres
    well_summ_parts.loc[:,'numparts_well']=numparts
    well_summ_parts.loc[:,'avglogRES_well']=meanlog10res
    #well_summ_parts.to_csv('well_summ_parts_{}_8284.csv'.format(new_sitenos[0]))
    well_summ_parts.to_csv(os.path.join('wellSumms/Redo','well_summ_parts_{}_11164.csv'.format(siteno)))
    print('# file finished: ', idx + 1)
    print(f"File: {filename} has finished!")
    print('Grazie!')
    #sys.exit()
    print("Thank you!")

print("All files have finished processing!")

sitenos: ['333224090114002', '333315090105301', '333315090105302', '333547090180301', '333547090180501', '333548090180601', '333611090161501', '333805090240501', '333824090320702', '333900090123702', '333900090123703']
Number of files: 11
Processing file: C:\github\map_gwage\MERASwd\Models\meras_2.2\layers_cal_wt_1.00\meras1\Shellmound\output\pld_33322409011400210204.csv
(56269, 24)
Reprojection completed
Start extraction.


C:\Anaconda3\envs\gis-2\lib\site-packages\pandas\core\arraylike.py:397: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Anaconda3\envs\gis-2\lib\site-packages\scipy\stats\_distn_infrastructure.py:1987: RuntimeWarning: invalid value encountered in subtract
  x = np.asarray((x - loc)/scale, dtype=dtyp)


Extraction complete for file:  C:\github\map_gwage\MERASwd\Models\meras_2.2\layers_cal_wt_1.00\meras1\Shellmound\output\pld_33322409011400210204.csv
Thank You!
numress: 56269
meanresdenom: 813.4535159333967
VARresWell: 1374.3324723386932
log normal
Name: 1
smeanres:  76.94408129329823
numlogress: 346
meanlogresdenom: 138.06143517340726
Name: 2
smeanres:  69.63748489714862
numlogress: 347
meanlogresdenom: 146.4405781762469
Name: 3
smeanres:  65.8033045175094
numlogress: 349
meanlogresdenom: 148.83181709950043
Name: 4
smeanres:  87.43737460373136
numlogress: 352
meanlogresdenom: 139.37314257582696
Name: 5
smeanres:  86.36466845404199
numlogress: 490
meanlogresdenom: 173.5610039483671
Name: 6
smeanres:  73.52962242141226
numlogress: 355
meanlogresdenom: 147.1048900560331
Name: 7
smeanres:  65.8483335692518
numlogress: 404
meanlogresdenom: 161.06506348512704
Name: 8
smeanres:  78.79962181525211
numlogress: 358
meanlogresdenom: 144.72607872705836
Name: 9
smeanres:  72.5314455537583
numlogre

Name: 95
smeanres:  95.32330403129957
numlogress: 449
meanlogresdenom: 126.86044948861947
Name: 96
smeanres:  67.10880892546759
numlogress: 351
meanlogresdenom: 149.2429976455092
Name: 97
smeanres:  42.89896114118003
numlogress: 328
meanlogresdenom: 199.74613209907596
Name: 98
smeanres:  78.51136426284027
numlogress: 363
meanlogresdenom: 145.81707803323434
Name: 99
smeanres:  51.64882206707228
numlogress: 415
meanlogresdenom: 206.39224437726736
Name: 100
smeanres:  119.29827778472905
numlogress: 349
meanlogresdenom: 100.5980699616567
Calculations for all particles completed!
len['333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '333224090114002', '33322409011400

(59890, 24)
Reprojection completed
Start extraction.


...

(115938, 24)
Reprojection completed
Start extraction.


C:\Anaconda3\envs\gis-2\lib\site-packages\pandas\core\arraylike.py:397: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Anaconda3\envs\gis-2\lib\site-packages\scipy\stats\_distn_infrastructure.py:1987: RuntimeWarning: invalid value encountered in subtract
  x = np.asarray((x - loc)/scale, dtype=dtyp)


Extraction complete for file:  C:\github\map_gwage\MERASwd\Models\meras_2.2\layers_cal_wt_1.00\meras1\Shellmound\output\pld_33390009012370310204.csv
Thank You!
numress: 115938
meanresdenom: 2606.7174600264116
VARresWell: 632.156669885081
log normal
Name: 1
smeanres:  94.37321875278887
numlogress: 366
meanlogresdenom: 138.30110591429326
Name: 2
smeanres:  72.77489263848099
numlogress: 367
meanlogresdenom: 148.7372870871489
Name: 3
smeanres:  90.83789251447091
numlogress: 263
meanlogresdenom: 129.36312254819356
Name: 4
smeanres:  71.46471772695305
numlogress: 370
meanlogresdenom: 148.75762539940504
Name: 5
smeanres:  59.8111474697775
numlogress: 370
meanlogresdenom: 155.3713669697504
Name: 6
smeanres:  71.02225280196741
numlogress: 368
meanlogresdenom: 150.01810749246093
Name: 7
smeanres:  65.11302065485175
numlogress: 354
meanlogresdenom: 152.23028350440757
Name: 8
smeanres:  87.4386799183415
numlogress: 265
meanlogresdenom: 131.11642188315057
Name: 9
smeanres:  78.8126829625794
numlogr

smeanres:  88.45659416583212
numlogress: 263
meanlogresdenom: 132.18172616394492
Name: 92
smeanres:  63.970314960027416
numlogress: 367
meanlogresdenom: 154.46477678075138
Name: 93
smeanres:  63.77808140294782
numlogress: 369
meanlogresdenom: 154.84124235035586
Name: 94
smeanres:  19.440779985684266
numlogress: 343
meanlogresdenom: 259.39403555031106
Name: 95
smeanres:  78.10899366138544
numlogress: 457
meanlogresdenom: 235.42260002811287
Name: 96
smeanres:  61.81704925629198
numlogress: 367
meanlogresdenom: 155.77394577324702
Name: 97
smeanres:  83.7348945137035
numlogress: 430
meanlogresdenom: 221.73813819976158
Name: 98
smeanres:  33.104071190376835
numlogress: 263
meanlogresdenom: 164.13547778891427
Name: 99
smeanres:  65.94574189336761
numlogress: 365
meanlogresdenom: 152.47670310702222
Name: 100
smeanres:  65.97445339043306
numlogress: 265
meanlogresdenom: 140.33210606993694
Name: 101
smeanres:  46.26725796410193
numlogress: 351
meanlogresdenom: 165.99515055036076
Name: 102
smean

smeanres:  49.028143492380515
numlogress: 350
meanlogresdenom: 162.7654433969488
Name: 196
smeanres:  33.827176402671554
numlogress: 364
meanlogresdenom: 185.54809952815148
Name: 197
smeanres:  52.24955236168003
numlogress: 367
meanlogresdenom: 164.43764361381068
Name: 198
smeanres:  43.86854775453757
numlogress: 369
meanlogresdenom: 171.37360047076027
Name: 199
smeanres:  55.43077187178873
numlogress: 350
meanlogresdenom: 157.74573620925108
Name: 200
smeanres:  60.26973368281462
numlogress: 351
meanlogresdenom: 154.146854370315
Calculations for all particles completed!
len['333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '333900090123703', '

INDX2=range(504)
INDX2

test = pd.read_csv('particle_1_v2.csv')
grpdf_test = pd.DataFrame(data=test, index=INDX2)
grpdf_test

In [9]:
#grpdf_test.to_csv('grpdf_1_test.csv', index=False)
#print('Thank You!')

test = pd.read_csv('grpdf_v2_1.csv')
VAR=(math.fsum(test['dfms']))/(496-1)  # Variance of raw resistivities along one particle's flowpath.
print("sum: ", math.fsum(test['dfms']))
print("Numlocs: ", 496)
print('VAR: ', VAR)

test

test2 = test.head(100)
test2

math.fsum(test2['dfms'])

VAR = (np.nansum(test['dfms'])/495)
VAR

type(VAR)

type(math.fsum(test2['dfms']))